# Example: Evolutive equilibrium calculations

This notebook will demonstrate the core use-case for FreeGSNKE: simulating the (time-dependent) evolution of Grad-Shafranov (GS) equilibria. In particular we will simulate a **vertical displacement event (VDE)**.

To do this, we need to:
- build the tokamak machine.
- instatiate a GS equilibrium (to be used as an initial condition for the evolutive solver).
- calculate a vertical instability growth rate for this equilibrium and carry out passive structure mode removal via a normal mode decomposition (i.e. removing modes that have little effect on the evolution). More details on this in a later notebook. 
- define time-dependent coil voltages, plasma current density profile parameters, and plasma resistivity.
- evolve the active coil currents, the total plasma current, and the equilbirium using these profile parameters and voltages by solving the circuit equations alongside the GS equation.

Refer to the paper by [Amorisco et al. (2024)](https://pubs.aip.org/aip/pop/article/31/4/042517/3286904/FreeGSNKE-A-Python-based-dynamic-free-boundary) for more details.

## Import packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import pickle
import time

## Build the machine

In [ ]:
# build machine
from freegsnke import build_machine
tokamak = build_machine.tokamak(
    active_coils_path=f"../machine_configs/MAST-U/MAST-U_like_active_coils.pickle",
    passive_coils_path=f"../machine_configs/MAST-U/MAST-U_like_passive_coils.pickle",
    limiter_path=f"../machine_configs/MAST-U/MAST-U_like_limiter.pickle",
    wall_path=f"../machine_configs/MAST-U/MAST-U_like_wall.pickle",
)

## Diverted plasma example

### Instantiate an equilibrium

In [ ]:
from freegsnke import equilibrium_update

eq = equilibrium_update.Equilibrium(
    tokamak=tokamak,
    Rmin=0.1, Rmax=2.0,   # Radial range
    Zmin=-2.2, Zmax=2.2,  # Vertical range
    nx=65,                # Number of grid points in the radial direction (needs to be of the form (2**n + 1) with n being an integer)
    ny=65,               # Number of grid points in the vertical direction (needs to be of the form (2**n + 1) with n being an integer)
    # psi=plasma_psi
)  

### Instatiate a profile object

In [ ]:
from freegsnke.jtor_update import Lao85

profiles = Lao85(
    eq=eq,                     # equilibrium object
    Ip=7.59e5,                 # plasma current
    fvac=-0.52,                # fvac = rB_{tor}
    alpha=[362685, 17696],     # Lao profiles parameters
    beta=[0.103, 0.753],       # Lao profiles parameters
    alpha_logic=True,
    beta_logic=True,
)

### Set coil currents
Here we set coil currents that create a diverted plasma (as seen in prior notebooks). 

In [ ]:
current_values = np.array([2766.10782056,   193.12768191,  4019.86873871,  4857.42357466,
        -726.99282376, -1696.92521166,   -77.7252396 ,   266.38207493,
         -73.01483716, -4169.86874271, -4518.28147417,   5.2  ])

for i, key in enumerate(tokamak.coil_names[0:12]):
    eq.tokamak.set_coil_current(coil_label=key, current_value=current_values[i])

### Instatiate the solver


In [ ]:
from freegsnke import GSstaticsolver
GSStaticSolver = GSstaticsolver.NKGSsolver(eq)    

### Call forward solver to find equilibrium 

In [ ]:
GSStaticSolver.solve(eq=eq, 
                     profiles=profiles, 
                     constrain=None, 
                     target_relative_tolerance=1e-8,
                     verbose=0
                     )

### Plot the initial equilibrium 

This will be the initial condition, much like in the other examples. 

In [ ]:
fig1, ax1 = plt.subplots(1, 1, figsize=(4, 8), dpi=80)
ax1.grid(True, which='both')
eq.plot(axis=ax1, show=False)
eq.tokamak.plot(axis=ax1, show=False)
ax1.set_xlim(0.1, 2.15)
ax1.set_ylim(-2.25, 2.25)
plt.tight_layout()

In [ ]:
from freegsnke import virtual_circuits

VCs = virtual_circuits.VirtualCircuitHandling()
VCs.define_solver(GSStaticSolver)

# define the descriptors function (it should return an array of values and take in an eq object)
def plasma_descriptors(eq):

    # inboard/outboard midplane radii
    RinRout = eq.innerOuterSeparatrix()

    # inboard midplane radius
    Rout = eq.innerOuterSeparatrix()[0]

    # find lower X-point
    # define a "box" in which to search for the lower X-point
    XPT_BOX = [[0.33, -0.88], [0.95, -1.38]]

    # mask those points
    xpt_mask = (
        (eq.xpt[:, 0] >= XPT_BOX[0][0])
        & (eq.xpt[:, 0] <= XPT_BOX[1][0])
        & (eq.xpt[:, 1] <= XPT_BOX[0][1])
        & (eq.xpt[:, 1] >= XPT_BOX[1][1])
    )
    xpts = eq.xpt[xpt_mask, 0:2].squeeze()
    if xpts.ndim > 1 and xpts.shape[0] > 1:
        opt = eq.opt[0, 0:2]
        dists = np.linalg.norm(xpts - opt, axis=1)
        idx = np.argmin(dists)  # index of closest point
        Rx, Zx = xpts[idx, :]
    else:
        Rx, Zx = xpts

    return np.array([RinRout[0], RinRout[1], Rx, Zx])

target_names = ["Rin", "Rout", "Rx", "Zx"]
target_values = plasma_descriptors(eq)

In [ ]:
# next we need to define which coils we wish to calculate the shape (finite difference) derivatives (and therefore VCs) for
print("All active coils :", eq.tokamak.coils_list[0:12])

# here we'll look at a subset of the coils and the targets
coils = ['PX', 'D1', 'D2', 'D3', 'Dp', 'D5', 'D6', 'D7', 'P4', 'P5']
print("Coils to calculate VCs for :", coils)

In [ ]:
# calculate the shape and VC matrices using the in-built method
VCs.calculate_VC(
    eq=eq,
    profiles=profiles,
    coils=coils,
    target_names=target_names,
    target_calculator=plasma_descriptors,
    starting_dI=None,    
    min_starting_dI=50,
    verbose=True,
    name="VC_for_lower_targets", # name for the VC
    )

In [ ]:
# these are the finite difference derivatives of the targets wrt the coil currents
shape_matrix = 1.0*VCs.VC_for_lower_targets.shape_matrix
print(f"Dimension of the shape matrix is {shape_matrix.shape} [no. of targets x no. of coils]. Has units [m/A]")

# these are the VCs corresponding to the above shape matrix
VCs_matrix = 1.0*VCs.VC_for_lower_targets.VCs_matrix
print(f"Dimension of the virtual circuit matrix is {VCs_matrix.shape} [no. of coils x no. of targets]. Has units [A/m].")


In [ ]:
np.round(VCs_matrix.T)

#### Prepare inputs for the PCS class

Here, we will set up the inputs (lists and dictionaries) for each of the classes in the FreeGSNKE PCS (e.g. plasma, shape, virtual circuits). 

The inputs for each category should be a dictionary of dictionaries, each of which are "waveforms" with a specific key name that needs to be set. In the following cell, we show what the format of these waveforms should be by displaying the "ip_ref" waveform (i.e. the reference waveform used for plasma current feedback control). This data will be linearly interpolated once the PCS class is initialised. Some inputs, typically those that require arrays to be specified at a particular time (or perhaps gains or machine constants), are "previous value" interpolated. The type of interploation used is indicated next to each input. 

These default settings can be used for loading in an existing MAST-U shot set up. Later on, we will demonstrate how inputs can be modified to test new scenarios (before running a simulation).

Please feel free to explore the setups of the following dictionaries to get familiar with them.

In [ ]:
# COIL NAMES AND SHAPE TARGET NAMES

# active coils must match the order they appear in FreeGSNKE machine description
active_coils = ['Solenoid', 'PX', 'D1', 'D2', 'D3', 'Dp', 'D5', 'D6', 'D7', 'P4', 'P5', 'P6']

# subset of active coils to be used for shape target control (i.e. exclude vertical control coil P6 here)
ctrl_coils = ['Solenoid', 'PX', 'D1', 'D2', 'D3', 'Dp', 'D5', 'D6', 'D7', 'P4', 'P5']

# specify the solenoid 
solenoid_coils=["Solenoid"]

# specify the vertical control coil
vertical_coils=["P6"]

# names of the shape parameters to control (defined later on)
ctrl_targets = ['Rin','Rout', 'Rx', 'Zx']

# names of the plasma targets (this will be clear later)
plasma_targets = ['plasma']

Explain what a waveform is supposed to look like:

{'times': array([-0.01485, -0.01465, -0.01445, ...,  0.99935,  0.99955,  0.99975]),
 'vals': array([ -1775.0575,  -1775.0575,  -1884.9923, ..., 750000.06  ,
        750000.06  , 750000.06  ], dtype=float32)}

In [ ]:
# choose some min amd max simulation times
tmin = 0.0
tmax = 0.5

In [ ]:
# PLASMA CATEGORY DATA

# build the dictionary which will be passed to the FreeGSNKE PCS (all keys below are required)
plasma_data = {}

# plasma current reference (keep constant)
plasma_data["ip_ref"] = {
    'times': np.array([tmin, tmax]),
    'vals': np.array([eq._profiles.Ip, eq._profiles.Ip])
    }

# plasma current blending (between FF at 0 and FB at 1) here we just do FB
plasma_data["ip_blend"] = {
    'times': np.array([tmin, tmax]),
    'vals': np.array([1, 1])
    }

# vloop reference (here we don't use a loop voltage as we're in FB mode only)
plasma_data["vloop_ff"] = {
    'times': np.array([tmin, tmax]),
    'vals': np.array([0, 0])
    }

# plasma current feedback proprtional gain
plasma_data["k_prop"] = {
    'times': np.array([tmin, tmax]),
    'vals': np.array([-5*4, -5*4])
    }

# plasma current feedback integral gain
plasma_data["k_int"] = {
    'times': np.array([tmin, tmax]),
    'vals': np.array([-50*2, -50*2])
    }

# mutual inductance between plasma and solenoid (not required as we don't use FF control here)
plasma_data["M_solenoid"] = {"times": np.array([0]), "vals": np.array([1.0])}  # not used as we're working in FB

plasma_data.keys()

In [ ]:
# SHAPE/DIVERTOR CATEGORY

# build the dictionary which will be passed to the FreeGSNKE PCS (all keys below are required)
shape_data = {}

# waveforms for Rin
shape_data["Rin"] = {}
shape_data["Rin"]["ff"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
shape_data["Rin"]["ref"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.28, 0.28])}
shape_data["Rin"]["blend"] = {"times": np.array([tmin, tmax]), "vals": np.array([1.0, 1.0])}
shape_data["Rin"]["k_prop"] = {"times": np.array([tmin, tmax]), "vals": np.array([120, 120])}
shape_data["Rin"]["k_int"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
shape_data["Rin"]["damping"] = {"times": np.array([tmin, tmax]), "vals": np.array([1.0, 1.0])}

# waveforms for Rout
shape_data["Rout"] = {}
shape_data["Rout"]["ff"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
shape_data["Rout"]["ref"] = {"times": np.array([tmin, tmax]), "vals": np.array([1.35, 1.35])}
shape_data["Rout"]["blend"] = {"times": np.array([tmin, tmax]), "vals": np.array([1.0, 1.0])}
shape_data["Rout"]["k_prop"] = {"times": np.array([tmin, tmax]), "vals": np.array([170, 170])}
shape_data["Rout"]["k_int"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
shape_data["Rout"]["damping"] = {"times": np.array([tmin, tmax]), "vals": np.array([1.0, 1.0])}

# waveforms for Rx
shape_data["Rx"] = {}
shape_data["Rx"]["ff"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
shape_data["Rx"]["ref"] = {"times": np.array([tmin, 0.05, 0.15, 0.35, 0.45, tmax]), "vals": np.array([0.57, 0.57, 0.5, 0.5, 0.57, 0.57])}
shape_data["Rx"]["blend"] = {"times": np.array([tmin, tmax]), "vals": np.array([1.0, 1.0])}
shape_data["Rx"]["k_prop"] = {"times": np.array([tmin, tmax]), "vals": np.array([120, 120])}
shape_data["Rx"]["k_int"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
shape_data["Rx"]["damping"] = {"times": np.array([tmin, tmax]), "vals": np.array([1.0, 1.0])}

# waveforms for Zx
shape_data["Zx"] = {}
shape_data["Zx"]["ff"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
shape_data["Zx"]["ref"] = {"times": np.array([tmin, tmax]), "vals": np.array([-1.24, -1.24])}
shape_data["Zx"]["blend"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
shape_data["Zx"]["k_prop"] = {"times": np.array([tmin, tmax]), "vals": np.array([120, 120])}
shape_data["Zx"]["k_int"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
shape_data["Zx"]["damping"] = {"times": np.array([tmin, tmax]), "vals": np.array([1.0, 1.0])}

shape_data.keys()

In [ ]:
# VIRTUAL CIRCUITS CATEGORY

# build the dictionary which will be passed to the FreeGSNKE PCS (all keys below are required)
circuits_data = {}

circuits_data["Rin"] = {
    "times": np.array([tmin, tmax]), 
    "vals": [
        np.array([0, 3.1394e+04,  8.5990e+03, -8.8300e+02, -1.2460e+03, -6.6690e+03, 1.2660e+03,  2.9770e+03,  3.5350e+03,  8.3110e+03, -7.8520e+03]),
        np.array([0, 3.1394e+04,  8.5990e+03, -8.8300e+02, -1.2460e+03, -6.6690e+03, 1.2660e+03,  2.9770e+03,  3.5350e+03,  8.3110e+03, -7.8520e+03]),
        ]
    }

circuits_data["Rout"] = {
    "times": np.array([tmin, tmax]), 
    "vals": [
        np.array([0, -9.9700e+02,  1.1790e+03,  3.9700e+02, -2.0000e+02, -1.8970e+03, -5.5500e+02, -2.2640e+03, -1.5860e+03, -1.9750e+03,  5.5800e+03]),
        np.array([0, -9.9700e+02,  1.1790e+03,  3.9700e+02, -2.0000e+02, -1.8970e+03, -5.5500e+02, -2.2640e+03, -1.5860e+03, -1.9750e+03,  5.5800e+03]),
        ]
    }

circuits_data["Rx"] = {
    "times": np.array([tmin, tmax]), 
    "vals": [
        np.array([0, -3.0021e+04,  3.3980e+03,  8.6900e+03,  5.8310e+03,  1.5858e+04, 2.4000e+01, -1.2100e+02, -2.2160e+03, -9.5290e+03,  1.3570e+03]),
        np.array([0, -3.0021e+04,  3.3980e+03,  8.6900e+03,  5.8310e+03,  1.5858e+04, 2.4000e+01, -1.2100e+02, -2.2160e+03, -9.5290e+03,  1.3570e+03]),
        ]
    }

circuits_data["Zx"] = {
    "times": np.array([tmin, tmax]), 
    "vals": [
        np.array([0, 3.7670e+03,  2.0328e+04,  1.0562e+04,  4.2560e+03,  2.2600e+03, -2.0450e+03, -5.5680e+03, -5.3160e+03, -9.8440e+03,  3.3750e+03]),
        np.array([0, 3.7670e+03,  2.0328e+04,  1.0562e+04,  4.2560e+03,  2.2600e+03, -2.0450e+03, -5.5680e+03, -5.3160e+03, -9.8440e+03,  3.3750e+03]),
        ]
    }

circuits_data["plasma"] = {
    "times": np.array([tmin, tmax]), 
    "vals": [
        np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        np.array([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        ]
    }

circuits_data["coil_order"] = ctrl_coils

# next, define any feedforward coil current drives on each control coil
zeros_dict = {"times": np.array([tmin, tmax]), "vals": np.array([0.0, 0.0])}
for coil in ctrl_coils: # linearly interpolated
    circuits_data[coil+"_ref"] = zeros_dict

circuits_data.keys()

In [ ]:
# SYSTEMS CATEGORY

# build the dictionary which will be passed to the FreeGSNKE PCS (all keys below are required)
systems_data = {}

# define the control coil current perturbations
for name in ctrl_coils: # linearly interpolated
    systems_data[name+"_pert"] = zeros_dict

# define the min and max coil current limits and ramp rate limits
systems_data["max_coil_curr_lims"] = {
    'times': [-2.6],
    'vals': [np.array([10000, 7000, 9000, 9000, 9000, 9000, 9000, 9000, 9000, 0, 0])],
    }

systems_data["min_coil_curr_lims"] = {
    'times': [-2.6],
    'vals': [np.array([-10000, -7000, -9000, -9000, -9000, -9000, -9000, -9000, -9000, -10000, -10000])],
    }

systems_data["max_coil_curr_ramp_lims"] = {
    'times': [-2.6],
    'vals': [1e12*np.ones(len(ctrl_coils))],
    }

systems_data.keys()

In [ ]:
# PF CATEGORY

# build the dictionary which will be passed to the FreeGSNKE PCS (all keys below are required)
pf_data = {}

# coil resistances
pf_data["R_matrix"] = {
    'times': [0.0],
    'vals': [tokamak.coil_resist[0:11]],
    }

# coil mutual inductances (on feedforward terms)
pf_data["M_FF_matrix"] = {
    'times': [0.0],
    'vals': [tokamak.coil_self_ind[0:11,0:11]],
    }

# coil mutual inductances (on feedback terms)
pf_data["M_FB_matrix"] = {
    'times': [0.0],
    'vals': [tokamak.coil_self_ind[0:11,0:11]],
    }


# gains on the coils (for feedback term)
pf_data["coil_gains"] = {
    'times': [0.0],
    'vals': [0.015*np.ones(len(ctrl_coils))],
    }

# limits on voltages in the coils
pf_data["coil_voltage_lims"] = {
    "times": np.array([-2.6]),
    'vals': [[2000,  750,  750,  750,  750,  750,  750,  750,  750,  750,  750]],
     } 

# limits on voltage ramp rates in the coils
pf_data["coil_voltage_slew_lims"] = {
    'times': [-2.6],
    'vals': [1e12*np.ones(len(ctrl_coils))],
    }
    

pf_data.keys()

In [ ]:
# VERTICAL CATEGORY DATA

# build the dictionary which will be passed to the FreeGSNKE PCS (all keys below are required)
vertical_data = {}

# plasma vertical position reference
vertical_data["z_ref"] = {"times": np.array([tmin, tmin+0.25, tmax]), "vals": np.array([0.0, 0.01, 0.01])}

# proportional gain
vertical_data["k_prop"] = {"times": np.array([tmin, tmax]), "vals": np.array([0.025, 0.025])}

# derivative gain
vertical_data["k_deriv"] = {"times": np.array([tmin, tmax]), "vals": np.array([5e-7, 5e-7])}

vertical_data.keys()

In [ ]:
# COIL ACTIVATIONS CATEGORY

# build the dictionary which will be passed to the FreeGSNKE PCS (all keys below are required)
coil_activation_data = {}

# coil activation times
default_coil_dict = {"times": np.array([tmin, tmax]), "vals": np.array([1.0, 1.0])}
for name in active_coils:
    coil_activation_data[name+"_activation"] = default_coil_dict

coil_activation_data

#### Initialise the FreeGSNKE PCS class

Now that the inputs have been prepared we can initialise the class and use in-built functions to view or plot the interpolated data in each control category.

In [ ]:
# FIRE UP THE PLASMA CONTROL SYSTEM
from freegsnke.control_loop.pcs import PlasmaControlSystem

PCS = PlasmaControlSystem(
    plasma_data=plasma_data,
    shape_data=shape_data,
    circuits_data=circuits_data,
    systems_data=systems_data,
    pf_data=pf_data,
    vertical_data=vertical_data,
    coil_activation_data=coil_activation_data,
    active_coils=active_coils,
    ctrl_coils=ctrl_coils,
    solenoid_coils=solenoid_coils,
    vertical_coils=vertical_coils,
    ctrl_targets=ctrl_targets,
    plasma_target=plasma_targets,
)

The following can be done for each of the different controllers.

Explain the plot colours. 

In [ ]:
# # view raw data
# PCS.PlasmaController.data

# # display raw data and interpolated values in a plot in the Plasma Controller
# PCS.PlasmaController.plot_data(tmin=tmin-0.05, tmax=tmax+0.05)

# display raw data and interpolated values in a plot for a specific shape target in the Shape Controller
PCS.ShapeController.plot_data(targ="Rx", tmin=0.0, tmax=1.0)

Now suppose you wish to develop a new scenario based on this shot setup data.

To modify the input data, simply edit the values in the `.data` attribute of the controller of interest and then call `.update_interpolants()`. This has to be done to ensure that the controller refreshes any stale interpolants from a prior initialisation.

Uncomment the code below to see how it works.

In [ ]:
# # suppose we wish to modify the (feedback) reference waveform for the "shape1" target
# PCS.ShapeController.data["Rin"]["ref"]

In [ ]:
# # update the data entry with your new waveform
# new_waveform = {"times": np.array([0.1, 0.2, 0.4, 0.6, 0.8, 0.9]), "vals": np.array([0.75, 1.25, 1.0, 1.0, 1.25, 0.75])}  # previous value interpolated
# PCS.ShapeController.data["Rin"]["ref"] = new_waveform

# # call the update function to refresh inteprolants (essential)
# PCS.ShapeController.update_interpolants()

# # plot to see new waveform
# PCS.ShapeController.plot_data(targ="Rin", tmin=0.0, tmax=1.0)

In [ ]:
# define the descriptors function (it should return an array of values and take in an eq object)
def plasma_descriptors(eq):

    # inboard/outboard midplane radii
    RinRout = eq.innerOuterSeparatrix()

    # find lower X-point
    # define a "box" in which to search for the lower X-point
    XPT_BOX = [[0.33, -0.88], [0.95, -1.38]]

    # mask those points
    xpt_mask = (
        (eq.xpt[:, 0] >= XPT_BOX[0][0])
        & (eq.xpt[:, 0] <= XPT_BOX[1][0])
        & (eq.xpt[:, 1] <= XPT_BOX[0][1])
        & (eq.xpt[:, 1] >= XPT_BOX[1][1])
    )
    xpts = eq.xpt[xpt_mask, 0:2].squeeze()
    if xpts.ndim > 1 and xpts.shape[0] > 1:
        opt = eq.opt[0, 0:2]
        dists = np.linalg.norm(xpts - opt, axis=1)
        idx = np.argmin(dists)  # index of closest point
        Rx, Zx = xpts[idx, :]
    else:
        Rx, Zx = xpts

    return np.array([RinRout[0], RinRout[1], Rx, Zx])

In [ ]:
from freegsnke import nonlinear_solve

stepping = nonlinear_solve.nl_solver(
    eq=eq, 
    profiles=profiles, 
    GSStaticSolver=GSStaticSolver,
    full_timestep=5e-4, 
    plasma_resistivity=1e-7,
    fix_n_vessel_modes=30,               
    plasma_descriptor_function=plasma_descriptors,
    )

In [ ]:
constant_voltages = (stepping.vessel_currents_vec*stepping.evol_metal_curr.coil_resist)[:stepping.evol_metal_curr.n_active_coils] 

#### Define evolutive simulation parameters
Next we set the key parameters for the evolutive simulation, including the time step and number of steps.

In [ ]:
# SOLVER SETUP

# set the number of time steps and time step size
n = 500
dt = 1e-3
dt_PCS = 1e-4
tmin = 0.0

# automatically calculates time vector
stepping.dt_step = dt
t_end = tmin + n*dt
times = np.arange(tmin, t_end, dt)

# (re-)initialise the dynamic solver with the initial eq and profiles
stepping.initialize_from_ICs(eq, profiles)

In [ ]:
# note that we use constant profile parameters and resistivity here but really they will be time-dependent


#### FPDT Simulation

In the following cell, we initialise lists/arrays to store any equilbirium related data we wish to view after the simulation. See the FreeGSNKE example notebook (example03 - extracting_equilibrium_quantites) for a list of these and how to extract them. 

Note that some quantities can be computationally costly to extract and have not been optimised for speed yet!

In [ ]:
# FREEGSNKE OUTPUT DATA STORAGE
dynamic_psi = np.zeros((stepping.eq1.psi().shape[0], stepping.eq1.psi().shape[1], len(times)))  
dynamic_limiter_flag = np.zeros(len(times))
dynamic_psi_boundary = np.zeros(len(times))
dynamic_xpts = []
dynamic_opts = []
dynamic_currents = np.zeros((len(stepping.currents_vec[:-1]),len(times)))
dynamic_ip = np.zeros(len(times))
dynamic_shape_targets = np.zeros((len(ctrl_targets),len(times)))
dynamic_timings = np.zeros(len(times))
dynamic_triangularity = np.zeros(len(times))


# STORAGE FOR PCS DATA
V_approved = np.zeros((len(ctrl_coils)+1,len(times)))
ip_hist = np.zeros(len(times))
T_err = np.zeros((len(ctrl_targets),len(times)))
T_hist = np.zeros((len(ctrl_targets),len(times)))
I_approved = np.zeros((len(ctrl_coils),len(times)))
coil_resists = np.zeros((len(active_coils),len(times)))
z_current = []

PCS_dip_dt = []
PCS_dT_dt = []
PCS_I_unapproved = []
PCS_I_approved = []


# INITIAL VALUES
dynamic_psi[:,:,0] = stepping.eq1.psi()
dynamic_limiter_flag[0] = stepping.eq1._profiles.flag_limiter
dynamic_psi_boundary[0] = stepping.eq1._profiles.psi_bndry
dynamic_xpts.append(stepping.eq1.xpt)
dynamic_opts.append(stepping.eq1.opt)
dynamic_currents[:,0] = stepping.currents_vec[:-1].copy()
dynamic_ip[0] = stepping.profiles1.Ip
dynamic_shape_targets[:,0] = plasma_descriptors(eq=stepping.eq1)
dynamic_triangularity[0] = eq.triangularity()
z_current.append(stepping.eq1.Zcurrent())
dynamic_jtor_norm = []

threshold = None

In [ ]:
# RUN FPDT SIMULATION
for i, t in enumerate(times[0:-1]):
    print("-----")
    print(f"t = {np.round(t,5)}s (step {i+1}/{n-1})")
    
    # start timer
    start_time = time.time()

    # initialise any historical quantities for PCS PID controllers
    if i == 0:
        ip_hist_prev = 0.0
        T_err_prev = np.zeros(len(ctrl_targets))
        T_hist_prev = np.zeros(len(ctrl_targets))
        I_approved_prev = eq.tokamak.getCurrentsVec()[0:11]  # must use starting currents here
        V_approved_prev = np.zeros(len(ctrl_coils))
        zipv_meas = 0.0
    else:
        ip_hist_prev=ip_hist[i-1].copy()
        T_err_prev=T_err[:,i-1].copy()
        T_hist_prev=T_hist[:,i-1].copy()
        I_approved_prev=I_approved[:,i-1].copy()
        V_approved_prev=V_approved[0:-1,i-1].copy()
        zipv_meas = ((z_current[-1]-z_current[-2])/dt)*dynamic_ip[i]
    
    # call the PCS class
    V_approved[:,i], ip_hist[i], T_err[:,i], T_hist[:,i], I_approved[:,i], coil_resists[:,i] = PCS.calculate_ctrl_voltages(
        t=t,                                        # simulation time
        dt=dt_PCS,                                      # PCS time step
        dt_simulator=dt,                            # simulator time step
        ip_meas=dynamic_ip[i],                      # measured plasma current
        ip_hist_prev=ip_hist_prev,                  # plasma controller integral term history
        T_meas=dynamic_shape_targets[:,i].copy(),   # measured shape parameters
        T_err_prev=T_err_prev,                      # shape controller proportional term history
        T_hist_prev=T_hist_prev,                    # shape controller integral term history
        I_approved_prev=I_approved_prev,            # approved PF currents from prior timestep
        I_meas=dynamic_currents[0:11,i].copy(),     # measured PF coil currents
        V_approved_prev=V_approved_prev,            # approved PF voltages from prior timestep
        zip_meas=z_current[i]*dynamic_ip[i],        # measured vertical position x plasma current
        zipv_meas=zipv_meas,                        # derivative of above
        active_coil_resists=tokamak.coil_resist[0:12].copy(),    # PF coil resistances
        verbose=False,                              # print some output
    )

    # track internal PCS quantities
    PCS_dip_dt.append(PCS.dip_dt)
    PCS_dT_dt.append(PCS.dT_dt)
    PCS_I_unapproved.append(PCS.I_unapproved)
    PCS_I_approved.append(PCS.I_approved)

    # V_approved[1:-1,i] = constant_voltages[1:-1]

    
    # extract plasma current density profile parameters (these are constant but can be time-dep)
    profile_params = {
            "alpha": profiles.alpha[0:2],
            "beta": profiles.beta[0:2],
            }

    # run freegsnke over the time step with the calculated voltages
    stepping.nlstepper(
        plasma_resistivity=1e-7,                          # assign plasma resistivity
        active_voltage_vec=V_approved[:,i],               # assign active voltages
        profiles_parameters=profile_params,               # assign profile parameters
        custom_active_coil_resistances=coil_resists[:,i], # assign active coil resistances
        linear_only=True,                                 # linear or nonlinear solve?
        target_relative_tol_currents=1e-2,                # relative tolerance in the currents required for convergence
        target_relative_tol_GS=1e-2,                      # relative tolerance in the plasma flux required for convergence
        working_relative_tol_GS=(1e-2)/2,                     # tolerance used when solving all static GS problems, expressed in terms of the change in the plasma flux due to 1 timestep of evolution
        verbose=False,                                    # print some output
        relinearise_threshold=threshold,                 # if the relative jtor norm change from lat linearisation is above threshold, relinearise around current equilibrium
        no_GS=False,                                       # do not solve GS at each time?
        )

    # stop timer
    end_time = time.time()

    # STORE DATA
    dynamic_psi[:,:,i+1] = stepping.eq1.psi()
    dynamic_limiter_flag[i+1] = stepping.eq1._profiles.flag_limiter
    dynamic_psi_boundary[i+1] = stepping.eq1._profiles.psi_bndry
    dynamic_xpts.append(stepping.eq1.xpt)
    dynamic_opts.append(stepping.eq1.opt)
    dynamic_currents[:,i+1] = stepping.currents_vec[:-1].copy()
    dynamic_ip[i+1] = stepping.currents_vec[-1] * stepping.plasma_norm_factor
    dynamic_shape_targets[:,i+1] = plasma_descriptors(stepping.eq1)
    dynamic_timings[i] = end_time - start_time
    dynamic_triangularity[i] = stepping.eq1.triangularity()
    z_current.append(stepping.eq1.Zcurrent())
    dynamic_jtor_norm.append(stepping.relinearise_criteria)

    # PRINT SOME STUFF
    print(f"      Ip = {np.round(dynamic_ip[i+1]/1000,1)} [kA]")
    print(f"      P6 Voltage = {np.round(V_approved[-1,i],3)} [V]")
    print(f"      P6 Current = {np.round(dynamic_currents[11,i+1],3)} [A]")
    print(f"      Z position = {np.round(z_current[i+1],5)} [m]")


In [ ]:


fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)

# --- references and masks ---
FB_reference = PCS.PlasmaController.interpolants['ip_ref'](times)
FF_reference = PCS.PlasmaController.interpolants['vloop_ff'](times)

blend = PCS.PlasmaController.interpolants['ip_blend'](times)

FB_mask = (blend > 0) & (np.abs(FB_reference) > 0)
FF_mask = (blend < 1) & (np.abs(FF_reference) > 0)

# --- shade FB regions (green) ---
on_regions = np.where(np.diff(FB_mask.astype(int)) != 0)[0] + 1
for seg_t, seg_state in zip(np.split(times, on_regions),
                            np.split(FB_mask, on_regions)):
    if np.all(seg_state):
        ax.axvspan(seg_t[0], seg_t[-1],
                   color='green', alpha=0.25,
                   label="FB active")

# --- shade FF regions (yellow) ---
on_regions = np.where(np.diff(FF_mask.astype(int)) != 0)[0] + 1
for seg_t, seg_state in zip(np.split(times, on_regions),
                            np.split(FF_mask, on_regions)):
    if np.all(seg_state):
        ax.axvspan(seg_t[0], seg_t[-1],
                   color='gold', alpha=0.25,
                   label="FF active")

# --- FreeGSNKE ---
ax.plot(times, dynamic_ip,
        color='navy', linewidth=1,
        marker='x', markersize=0,
        label="FreeGSNKE")

# --- FB reference ---
ax.plot(times[FB_mask], FB_reference[FB_mask],
        color='k', linestyle='--',
        linewidth=1.5,
        label="FB reference")

ax.set_xlabel(r"Shot time [$s$]")
ax.set_ylabel("Plasma current [A]")
ax.grid()

# deduplicate legend entries
handles, labels = ax.get_legend_handles_labels()
ax.legend(dict(zip(labels, handles)).values(),
          dict(zip(labels, handles)).keys())

ax.set_ylim([7.54e5, 7.64e5])
fig.suptitle("Plasma Current", y=0.98)
plt.tight_layout()
plt.show()


In [ ]:


fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)

# --- FB reference ---
FB_reference = PCS.VerticalController.interpolants['z_ref'](times)

# --- shade FB-active region (always on here) ---
ax.axvspan(times[0], times[-1],
           color='green', alpha=0.2,
           label="FB active")

# --- references ---
ax.plot(times, FB_reference,
        color='k', linestyle='--',
        linewidth=1.5,
        label="FB reference")

# --- FreeGSNKE ---
ax.plot(times, z_current,
        color='navy', linewidth=1,
        marker='x', markersize=0,
        label="FreeGSNKE")

ax.set_xlabel(r"Shot time [$s$]")
ax.set_ylabel("Vertical position [m]")
ax.grid()
ax.legend()

fig.suptitle("Vertical Position", y=0.98)
plt.tight_layout()
plt.show()


In [ ]:


ntarg = len(ctrl_targets)

fig, axes = plt.subplots(
    nrows=ntarg,
    ncols=1,
    figsize=(8, 12),
    dpi=80,
    sharex=True
)

for j, targ in enumerate(ctrl_targets):
    ax = axes[j]

    # --- references and masks ---
    FF_reference = PCS.ShapeController.interpolants[targ]['ff'](times)
    FB_reference = PCS.ShapeController.interpolants[targ]['ref'](times)

    FF_mask = (
        (PCS.ShapeController.interpolants[targ]['blend'](times) < 1)
        & (np.abs(PCS.ShapeController.interpolants[targ]['ff'].derivative()(times)) > 0)
    )

    FB_mask = (
        (PCS.ShapeController.interpolants[targ]['blend'](times) > 0)
        & (np.abs(FB_reference) > 0)
    )

    # --- shade FB regions (green) ---
    on_regions = np.where(np.diff(FB_mask.astype(int)) != 0)[0] + 1
    for seg_t, seg_state in zip(np.split(times, on_regions),
                                np.split(FB_mask, on_regions)):
        if np.all(seg_state):
            ax.axvspan(seg_t[0], seg_t[-1], color='green', alpha=0.25,
                       label="FB active" if j == 0 else None)

    # --- shade FF regions (yellow) ---
    on_regions = np.where(np.diff(FF_mask.astype(int)) != 0)[0] + 1
    for seg_t, seg_state in zip(np.split(times, on_regions),
                                np.split(FF_mask, on_regions)):
        if np.all(seg_state):
            ax.axvspan(seg_t[0], seg_t[-1], color='gold', alpha=0.25,
                       label="FF active" if j == 0 else None)

    # --- references ---
    ax.plot(times[FB_mask], FB_reference[FB_mask],
            color='k', linestyle='--', linewidth=1.5,
            label="FB reference" if j == 0 else None)

    if np.any(FF_mask):
        offset = interpolants[targ + '_meas'](times[FF_mask][0])
        ax.plot(times[FF_mask], FF_reference[FF_mask] + offset,
                color='r', linestyle='--', linewidth=1.5,
                label="FF reference" if j == 0 else None)

    # --- FreeGSNKE ---
    ax.plot(times, dynamic_shape_targets[j, :],
            color='navy', linewidth=1,
            marker='x', markersize=0,
            label="FreeGSNKE" if j == 0 else None)

    ax.set_ylabel(f"{ctrl_targets[j]} [m]")
    ax.grid()
    ax.set_ylim([np.min(FB_reference)-0.02, np.max(FB_reference)+0.02])

# shared x label
axes[-1].set_xlabel(r"Shot time [$s$]")

# legend once
axes[0].legend(ncol=4, loc="upper right")

fig.suptitle("Shape parameters", y=0.995, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:


fig, ax = plt.subplots(
    nrows=1,
    ncols=1,
    figsize=(8, 4),
    dpi=80
)

# --- FreeGSNKE ---
ax.plot(times[0:-1], dynamic_triangularity[0:-1],
        color='navy', linewidth=1,
        marker='x', markersize=3,
        label="FreeGSNKE")

ax.set_xlabel(r"Shot time [$s$]")
ax.set_ylabel("Triangularity")
ax.grid()
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:


ncoils = len(active_coils)

fig, axes = plt.subplots(
    nrows=ncoils,
    ncols=2,
    figsize=(18, 45),
    dpi=80,
    sharex=True
)

for j, coil in enumerate(active_coils):
    axV = axes[j, 0]  # voltage column
    axI = axes[j, 1]  # current column

    # VOLTAGES (left)
    # if coil not in ["P6"]:
    #     vlim = PCS.PFController.data['coil_voltage_lims']['vals'][0][j]
    #     axV.hlines([-vlim, vlim], times[0], times[-1],
    #                colors='k', linestyles='--', linewidth=1.2,
    #                label="Coil limits" if j == 0 else None)

    axV.plot(times[0:-1], V_approved[j, 0:-1],
             color='navy', linewidth=1,
             marker='x', markersize=0,
             label="FreeGSNKE")

    axV.set_ylabel(f'{coil} voltage [V]')
    axV.grid()
    if j == 0:
        axV.legend()

    # CURRENTS (right)
    # if coil not in ["P6"]:
    #     imin = PCS.SystemsController.data['min_coil_curr_lims']['vals'][0][j]
    #     imax = PCS.SystemsController.data['max_coil_curr_lims']['vals'][0][j]
    #     axI.hlines([imin, imax], times[0], times[-1],
    #                colors='k', linestyles='--', linewidth=1.2,
    #                label="Coil limits" if j == 0 else None)

    axI.plot(times, dynamic_currents[j, :],
             color='navy', linewidth=1,
             marker='x', markersize=0,
             label="FreeGSNKE")

    axI.set_ylabel(f'{coil} current [A]')
    axI.grid()
    if j == 0:
        axI.legend()

# x-labels only on bottom row
axes[-1, 0].set_xlabel(r'Shot time [$s$]')
axes[-1, 1].set_xlabel(r'Shot time [$s$]')

# column titles
axes[0, 0].set_title("PF Coil Voltages")
axes[0, 1].set_title("PF Coil Currents")

plt.tight_layout()
plt.show()


In [ ]:
# # SAVE AN ANIMATION OF THE SIMULATION
# %matplotlib inline

# import matplotlib.animation as animation

# plt.rcParams['figure.dpi'] = 90

# fig1, ax1 = plt.subplots(1, 1, figsize=(8, 8))

# # Plot static wall / tokamak background
# eq.tokamak.plot(axis=ax1, show=False)
# ax1.plot(tokamak.wall.R, tokamak.wall.Z, color='k', linewidth=1.2)
# ax1.grid(True, which='both', alpha=0.5)
# ax1.set_aspect('equal')
# ax1.set_xlabel(r'Major radius, $R$ $[m]$')
# ax1.set_ylabel(r'Height, $Z$ $[m]$')
# ax1.set_xlim(0.05, 2.15)
# ax1.set_ylim(-2.25, 2.25)

# # Determine contour levels from entire dynamic_psi
# min_psi = np.min(dynamic_psi)
# max_psi = np.max(dynamic_psi)
# levels = np.linspace(min_psi, max_psi, 40)

# # --- Storage for dynamic artists ---
# contour_artists = []
# scatter_artists = []

# # --- Update function ---
# def update(i):
    
#     global contour_artists, scatter_artists

#     # Remove previous dynamic artists
#     for c in contour_artists + scatter_artists:
#         if isinstance(c, list):
#             for coll in c:
#                 coll.remove()
#         else:
#             c.remove()
#     contour_artists = []
#     scatter_artists = []

#     ax1.set_title(rf"$t$ = {np.round(times[i],3)}")

#     # Main psi contours
#     c1 = ax1.contour(eq.R, eq.Z, dynamic_psi[:,:,i], levels=levels, alpha=0.8, cmap='viridis')
#     contour_artists.append(c1)

#     # plasma boundary
#     c2 = ax1.contour(eq.R, eq.Z, dynamic_psi[:,:,i],
#                          levels=[dynamic_psi_boundary[i]],
#                          linestyles="-", colors='r', linewidths=1.4)
#     contour_artists.append(c2)

#     # adds separatrix of primary X-point if plasma limited
#     if dynamic_limiter_flag[i]:
#         c3 = ax1.contour(eq.R, eq.Z, dynamic_psi[:,:,i],
#                             levels=[dynamic_xpts[i][0,2]],
#                             linestyles="--", colors='k', linewidths=1.4)
#         contour_artists.append(c3)


#     # X-points and O-points
#     sc1 = ax1.scatter(dynamic_xpts[i][:,0], dynamic_xpts[i][:,1], color='r', marker='x', s=30)
#     sc2 = ax1.scatter(dynamic_opts[i][:,0], dynamic_opts[i][:,1], color='g', marker='2', s=30)

#     # indicate which is the primary X-point more clearly
#     sc3 = ax1.scatter(dynamic_xpts[i][0,0], dynamic_xpts[i][0,1], color='r', marker='x', s=60)

#     scatter_artists.extend([sc1, sc2, sc3])

#     return contour_artists + scatter_artists

# # --- Animation ---
# num_frames = len(times)
# frames_to_plot = range(0, num_frames, 5)
# fps = int(len(frames_to_plot)/10)
# ani = animation.FuncAnimation(fig1, update, frames=frames_to_plot, interval=1, blit=True)

# # save to video (much faster & smaller than GIF)
# ani.save(f"data/animated_equilibrium.mp4", writer="ffmpeg", fps=fps)
